# codexdojo

> Pre-baked Codex sessions that demonstrate a clean llmdojo round

In [ ]:
#| default_exp codexdojo

In [ ]:
#| export
import asyncio, json, os, re, sys, uuid
from importlib.resources import files
from fastcore.utils import *
from llmsurgery.oai import *
from llmsurgery.ipynb import read_ipynb, write_ipynb
from llmdojo.rules import _state_root, doced

In [ ]:
from fastcore.test import *
import tempfile, shutil

## Capturing a clean round

Codex stores Responses API items rather than Claude transcript records. A capture still has the same shape: isolate one successful active round, remove reasoning and host polling noise, and regenerate stable ids.

In [ ]:
#| export
def _items(src):
    "Responses items from either rollout records or an item sequence"
    src = L(src)
    return response_items(src) if any(x.get('type')=='response_item' for x in src) else L(obj2dict(x) for x in src)

def _out_text(item):
    out = item.get('output','')
    if isinstance(out,str): return out
    return '\n'.join(x.get('text','') for x in out if x.get('type') in ('input_text','output_text'))

def _logical(item): return parse_exec(item.get('input','')) if item.get('type')=='custom_tool_call' else None

def _cell(item):
    p = _logical(item)
    src = p.arguments.get('chars','') if p and p.name=='tools.write_stdin' else ''
    if src.startswith('--\n'):
        lines = src.splitlines()
        if lines[-1].startswith('--'): src = '\n'.join(lines[1:-1])+'\n'
    return src

### Deferred host calls

A slow terminal interaction may be recorded as an initial `custom_tool_call_output`, followed by a host-level `wait` function call carrying the real result. `collapse_deferred` restores the simpler logical call/result pair that belongs in a worked example.

In [ ]:
#| export
def collapse_deferred(src):
    "Custom call/result pairs, replacing host-level deferred waits with their final result"
    items,out = _items(src),[]
    cis = [i for i,x in enumerate(items) if x.get('type')=='custom_tool_call']
    for n,i in enumerate(cis):
        call,end = items[i],cis[n+1] if n+1<len(cis) else len(items)
        seg = items[i+1:end]
        result = first(x for x in seg if x.get('type')=='custom_tool_call_output' and x.get('call_id')==call.get('call_id'))
        if not result: continue
        if 'Script running with cell ID' in _out_text(result):
            waits = [x for x in seg if x.get('type')=='function_call' and x.get('name')=='wait']
            if waits:
                final = first(x for x in seg if x.get('type')=='function_call_output' and x.get('call_id')==waits[-1].get('call_id'))
                if final: result = codex_custom_output(call['call_id'], final.get('output',''))
        out += [call,result]
    return L(out)

In [ ]:
c = codex_custom_call('tools.write_stdin', dict(session_id=7, chars='1+1\n'), 'call_c')
wrapped = codex_custom_call('tools.write_stdin',dict(session_id=7,chars='--\ndojo_score()\n--abcde\n'),'call_wrapped')
test_eq(_cell(wrapped),'dojo_score()\n')
direct = [c,codex_custom_output('call_c','2')]
test_eq(collapse_deferred(direct), direct)
test_eq(collapse_deferred([*direct,c]),direct)
wait = codex_call('wait', dict(cell_id='9'), 'call_w')
deferred = [c,codex_custom_output('call_c','Script running with cell ID 9'),wait,codex_output('call_w','2')]
back = collapse_deferred(deferred)
test_eq(_out_text(back[1]), '2')
test_eq(back[1]['call_id'], 'call_c')

### Selecting calls

`pick_calls` selects native custom calls by logical tool name and either an argument subset or kernel-cell source. Exact cell matches win; prefix matching handles long cells without copying their whole source into the curation notebook.

In [ ]:
#| export
def pick_calls(
    src, # Rollout records or Responses items
    *specs, # `(logical_name, cell_source_or_argument_subset)` pairs
):
    "The last matching custom call/result pair for each spec, in spec order"
    pairs = collapse_deferred(src)
    calls = [(pairs[i],pairs[i+1],_logical(pairs[i])) for i in range(0,len(pairs),2)]
    out = []
    for name,want in specs:
        cand = [(c,r,p) for c,r,p in calls if p and p.name==name]
        if isinstance(want,dict): cand = [(c,r,p) for c,r,p in cand if all(p.arguments.get(k)==v for k,v in want.items())]
        else:
            exact = [(c,r,p) for c,r,p in cand if p.arguments.get('chars','')==want]
            cand = exact or [(c,r,p) for c,r,p in cand if p.arguments.get('chars','').startswith(want)]
        if not cand: raise ValueError(f'no custom call matches {name} {want!r}')
        out += cand[-1][:2]
    return L(out)

In [ ]:
picked = pick_calls(direct, ('tools.write_stdin','1+1\n'))
test_eq(picked, direct)
picked

`pick_turns` is the common clikernel shorthand: its selectors are cell sources and its results remain native `tools.write_stdin` pairs.

In [ ]:
#| export
def pick_turns(
    src, # Rollout records or Responses items
    *cells, # Kernel cell sources: matched exactly, else as a prefix; the last match wins
):
    "The write_stdin call/result pairs for `cells`, in that order"
    return pick_calls(src, *((('tools.write_stdin',c) for c in cells)))

The remaining examples use a minimal complete round: one `dojo_start()` call and one clean `dojo_score()` result.

In [ ]:
mini = [
    codex_custom_call('tools.write_stdin',dict(session_id=7,chars='dojo_start()\n'),'call_start'),
    codex_custom_output('call_start','dojo card'),
    codex_custom_call('tools.write_stdin',dict(session_id=7,chars='dojo_score(bash_calls=0)\n'),'call_score'),
    codex_custom_output('call_score','Clean round. Completion id: ab12')]

In [ ]:
test_eq([_cell(x) for x in pick_turns(mini,'dojo_start()\n')[::2]], ['dojo_start()\n'])

### Finding the round

`dojo_round` takes the last complete start-to-score span. Earlier completed receipts and unrelated calls elsewhere in a long rollout are ignored.

In [ ]:
#| export
def dojo_round(src):
    "The last complete dojo_start()/dojo_score() custom-tool span"
    pairs = collapse_deferred(src)
    calls = pairs[::2]
    starts = [i for i,c in enumerate(calls) if re.match(r'^dojo_start\(\s*\)\s*$', _cell(c))]
    if not starts: raise ValueError('no dojo_start() call')
    start = starts[-1]
    scores = [i for i,c in enumerate(calls[start:],start) if re.match(r'^dojo_score\(', _cell(c))]
    if not scores: raise ValueError('no dojo_score() after dojo_start()')
    return pairs[start*2:(scores[0]+1)*2]

In [ ]:
round_items = dojo_round(mini)
test_eq(len(round_items), 4)
[_cell(x) for x in round_items[::2]]

`curate_dojo` makes the chosen span reproducible by regenerating response-item and paired call ids with a stable salt.

In [ ]:
#| export
def curate_dojo(src):
    "The reproducible custom-tool items for the last complete dojo round"
    return reid_items(dojo_round(src), 'codexdojo')

`dojo_cid` reads the four-character completion receipt from the clean score output; launch-time registration makes the baked round valid in a resumed session.

In [ ]:
#| export
def dojo_cid(items):
    "The completion id recorded in the clean-round output"
    return first(m[1] for x in items if (m:=re.search(r'Completion id: ([0-9a-f]{4})', item_txt(x))))

In [ ]:
test_eq(dojo_cid(curate_dojo(mini)), 'ab12')

### Gating the capture

A template is accepted only when it has one start, one first-try score, paired results, no execution errors, and an output explicitly reporting a clean round.

In [ ]:
#| export
GATE_FORBID = ('recording a demo', 'these notes')

def is_clean(
    items, # A curated dojo round
    forbid=GATE_FORBID, # Phrases that must not appear in visible assistant text
):
    "Why a captured round cannot become a template; empty means clean"
    items = _items(items)
    calls = [x for x in items if x.get('type')=='custom_tool_call']
    cells = [_cell(x) for x in calls]
    probs = []
    if (n:=sum(bool(re.match(r'^dojo_start\(\s*\)\s*$', c)) for c in cells)) != 1: probs.append(f'{n} dojo_start calls')
    if (n:=sum(bool(re.match(r'^dojo_score\(', c)) for c in cells)) != 1: probs.append(f'{n} dojo_score calls (a clean round scores once, first try)')
    if any(re.match(r'^dojo_(?:redo|resume)\(', c) for c in cells): probs.append('needed a redo')
    if not any('Clean round' in _out_text(x) for x in items if x.get('type')=='custom_tool_call_output'): probs.append('no clean score')
    if (n:=sum('\nTraceback (most recent call last):' in _out_text(x) for x in items if x.get('type')=='custom_tool_call_output')): probs.append(f'{n} failed tool calls')
    ids = {x.get('call_id') for x in items if x.get('type')=='custom_tool_call_output'}
    if (n:=sum(x.get('call_id') not in ids for x in calls)): probs.append(f'{n} calls without results')
    visible = ' '.join(item_txt(x) for x in items if x.get('type')=='message' and x.get('role')=='assistant')
    if (bad:=[s for s in forbid if s in visible]): probs.append(f"script leaked into visible text: {', '.join(bad)}")
    return probs

In [ ]:
test_eq(is_clean(mini), [])

doc_output = codex_custom_output('call_doc', 'Exceptions render as `<error>` traceback.')
test_eq(is_clean([*mini,doc_output]), [])
failed_output = codex_custom_output('call_failed', '\nTraceback (most recent call last):')
test_eq(is_clean([*mini,failed_output]), ['1 failed tool calls'])

### Capturing from a live session

`capture_current` remains useful when a clean round has already happened in an interactive Codex thread. It keeps the contiguous native `exec` pairs from the clikernel launch through the first clean score, rather than depending on a fixed list of bootstrap cells.

The capture is gated before it is converted into the review dialog. A supplied `docs` list records the child thread's actual documented-tool state; an interactive capture defaults to the current thread's state.

In [ ]:
#| export
def _capture_span(src):
    "Native custom-call pairs from the last clikernel launch through its first dojo score"
    pairs = collapse_deferred(src)
    calls,cells = pairs[::2],[_cell(x) for x in pairs[::2]]
    starts = [i for i,c in enumerate(cells) if re.match(r'^dojo_start\(\s*\)\s*$',c)]
    if not starts: raise ValueError('no dojo_start() call')
    start = starts[-1]
    scores = [i for i,c in enumerate(cells[start:],start) if re.match(r'^dojo_score\(',c)]
    if not scores: raise ValueError('no dojo_score() after dojo_start()')
    launches = [i for i,c in enumerate(calls[:start+1]) if (p:=_logical(c)) and p.name=='tools.exec_command' and p.arguments.get('cmd')=='clikernel']
    if not launches: raise ValueError('no clikernel launch before dojo_start()')
    return pairs[launches[-1]*2:(scores[0]+1)*2]

def capture_current(
    ref=None, # Codex thread UUID; current thread if None
    d=None, # Template store dir; `TMPL_DIR` if None
    docs=None, # Names doc()'d in the captured thread; current doc-state if None
):
    "Gate an existing Codex thread's clean round and store its native template"
    selected = reid_items(_capture_span(response_items(load_rollout(ref))),'codexdojo')
    if probs:=is_clean(selected): raise ValueError('; '.join(probs))
    docs = doced() if docs is None else docs
    dlg = mk_template(selected,doced=docs)
    save_template(dlg2items(dlg),d,doced=docs)
    write_ipynb(dlg,Path(d or TMPL_DIR)/'template.ipynb')
    return dlg

In [ ]:
fake_boot = [codex_custom_call('tools.exec_command',dict(cmd='clikernel'),'call_launch'),
    codex_custom_output('call_launch','{"session_id":99}')]
span = _capture_span([*fake_boot,*mini])
test_eq(span,[*fake_boot,*mini])
test_eq([_logical(x).name for x in span[::2]],['tools.exec_command','tools.write_stdin','tools.write_stdin'])
test_eq(is_clean(span),[])

## Template dialog

A live capture can be curated without replaying the whole thread. `pick_turns` selects cells exactly, falling back to prefix matching, while `pick_calls` also handles the initial `tools.exec_command` launch.

### Building the review dialog

`mk_template` adds the synthetic user prompt and short framing messages around already selected native tool pairs. Disabling tool truncation preserves every receipt byte-for-byte for rebuilding.

In [ ]:
#| export
TMPL_PROMPT = "Bootstrap and complete the dojo and tell me when you're ready."
OPENING = 'Bootstrapping: I launch clikernel, read the tooling docs, check the catalog, then take the dojo.'

def mk_template(
    picked, # Native custom call/result pairs
    opening=OPENING, # Reply text preceding the first tool call
    closing="OK I'm ready.", # Reply text ending the round
    doced=None, # Names doc()'d during the round
):
    "A template dialog with native tool output left untruncated"
    items = [codex_msg('user',TMPL_PROMPT),codex_msg('assistant',opening),*picked,codex_msg('assistant',closing)]
    dlg = items2dlg(items,'codexdojo_template',mx=None)
    dlg.meta['llmdojo'] = dict(doced=list(doced or []))
    return dlg

In [ ]:
tdlg = mk_template(mini, opening='Practice first.', doced=['doc','rg'])
test_eq(tdlg.meta, dict(llmdojo=dict(doced=['doc','rg'])))
test_eq(tdlg.messages[0].content, TMPL_PROMPT)
back = dlg2items(tdlg)
call = first(x for x in back if x.get('type')=='custom_tool_call')
test_eq(parse_exec(call['input']).arguments['chars'], 'dojo_start()\n')

The packaged dialog is the fallback on a machine with no local template store. Read that reviewed artifact through the same conversion path and reject it if its calls, score, or completion receipt no longer survive the round trip.

In [ ]:
packaged = read_ipynb(str(files('llmdojo')/'dojo_data'/'codexdojo_template.ipynb'))
packaged_items = dlg2items(packaged)
assert dojo_cid(packaged_items)
test_eq(is_clean(packaged_items), [])

## Template store

The store keeps native Responses items plus the clean-round completion id, dojo version, and documented-tool state. The packaged notebook is only the portable source for a new machine's first store.

In [ ]:
#| export
TMPL_DIR = _state_root()/'codex_template'

def _dojo_v():
    from llmdojo.dojo import dojo_version
    return dojo_version()

`save_template` writes native items and the metadata needed to replay them: completion id, dojo version, and documented-tool state.

In [ ]:
#| export
def save_template(
    items, # Curated native template items
    d=None, # Store dir; `TMPL_DIR` if None
    doced=None, # Names doc()'d during the baked round; current doc-state if None
):
    "Write the template and its metadata to the store"
    if doced is None:
        from llmdojo.rules import doced as _cur
        doced = _cur()
    d = Path(d or TMPL_DIR)
    d.mkdir(parents=True,exist_ok=True)
    (d/'template.jsonl').write_text(''.join(json.dumps(obj2dict(x))+'\n' for x in items))
    (d/'meta.json').write_text(json.dumps(dict(cid=dojo_cid(items),v=_dojo_v(),doced=list(doced))))

`load_template` reads that pair of files without converting the items through another representation.

In [ ]:
#| export
def load_template(
    d=None, # Store dir; `TMPL_DIR` if None
):
    "The stored native items and metadata"
    d = Path(d or TMPL_DIR)
    return [json.loads(x) for x in (d/'template.jsonl').read_text().splitlines()],json.loads((d/'meta.json').read_text())

In [ ]:
store = Path(tempfile.mkdtemp())
save_template(dlg2items(tdlg),store,doced=['doc','rg'])
stored,meta = load_template(store)
test_eq(meta,dict(cid='ab12',v=_dojo_v(),doced=['doc','rg']))
test_eq(dojo_cid(stored),'ab12')

`build_template` is the reverse review path: convert an edited dialog to native items, regenerate ids, and update the store.

In [ ]:
#| export
def build_template(
    src, # Path to a template dialog
    d=None, # Store dir; `TMPL_DIR` if None
):
    "Convert a reviewed template dialog back into the native store"
    dlg = read_ipynb(str(src))
    items = reid_items(dlg2items(dlg),'codexdojo')
    save_template(items,d,doced=nested_idx(dlg.meta,'llmdojo','doced') or [])

In [ ]:
review = store/'review.ipynb'
write_ipynb(tdlg,review)
rebuilt = store/'rebuilt'
build_template(review,rebuilt)
test_eq(dojo_cid(load_template(rebuilt)[0]),'ab12')

## Launching

Creating a dojo session starts a Codex thread, injects the native template, registers its completion receipt, and seeds the documented-tool state. Appending performs the same injection at the tail of an existing thread after compaction.

In [ ]:
#| export
def _load_reg(d):
    "Load the store, building it from the packaged dialog on first use"
    if not (Path(d or TMPL_DIR)/'template.jsonl').exists(): build_template(files('llmdojo')/'dojo_data'/'codexdojo_template.ipynb',d)
    items,meta = load_template(d)
    if meta['v'] != _dojo_v(): print(f"codexdojo: template built under {meta['v']} but installed tooling is {_dojo_v()}; rebuild the reviewed dialog with codexdojo --build <dialog>.")
    from llmdojo.dojo import register_completion
    register_completion(meta['cid'],meta['v'])
    return items,meta

def _seed_doced(sid,names,merge=False):
    "Make the baked round's documented-tool state true for thread `sid`"
    d = _state_root()/'doced'
    d.mkdir(parents=True,exist_ok=True)
    f = d/f'{sid}.json'
    if merge and f.exists(): names = set(names) | set(json.loads(f.read_text()))
    f.write_text(json.dumps(sorted(names)))

### Starting a thread

`prep_dojo` creates a persisted Codex thread, injects a freshly re-id'd copy of the template, registers its completion, and seeds the matching documentation receipts.

In [ ]:
#| export
async def prep_dojo(
    cwd=None, # Project to start in; current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
    codex_home=None, # Alternate Codex home, primarily for tests
):
    "Create a thread containing the template and return its id"
    items,meta = _load_reg(d)
    async with CodexAppServer(codex_home=codex_home) as app:
        thread = await app.create_thread(reid_items(items,'launch:'+uuid.uuid4().hex),cwd=cwd)
    if ds:=meta.get('doced'): _seed_doced(thread.id,ds)
    return thread.id

### Refreshing after compaction

`append_dojo` appends a fresh copy of the worked round to an existing thread and changes only its synthetic prompt to explain why the exercise is repeating.

In [ ]:
#| export
APPEND_PROMPT = "We've compacted - run the dojo round again and tell me when you're ready."

def _prompt_text(item,text):
    item = obj2dict(item)
    item['content'] = [dict(type='input_text',text=text)]
    return item

async def append_dojo(
    sid=None, # Thread id to append to; newest thread for the project if None
    cwd=None, # Project directory; current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
    codex_home=None, # Alternate Codex home
):
    "Append the template to an existing thread and return its id"
    items,meta = _load_reg(d)
    sid,path = resolve_thread(sid,codex_home or CODEX_HOME) if sid else project_thread(cwd or '.',codex_home or CODEX_HOME)
    tail = load_recs(path)
    items = reid_items(items,f'append:{sid}:{len(tail)}')
    items[0] = _prompt_text(items[0],APPEND_PROMPT)
    async with CodexAppServer(codex_home=codex_home) as app: await app.append_thread(sid,items)
    _seed_doced(sid,meta.get('doced') or [],merge=True)
    return sid

`append_dojo` is the post-compaction repair. Native compaction replaces the worked history with a summary, so appending restores the demonstrations at the tail and merges their documented-tool state. Re-salting ids makes repeated repairs safe.

In [ ]:
#| eval: false
# App-server integration: isolated CODEX_HOME, no model turn.
with tempfile.TemporaryDirectory() as td:
    home,proj = Path(td),Path(td)/'project'
    proj.mkdir()
    sid = await prep_dojo(proj,store,home)
    back = thread2dlg(sid,home,mx=None)
    assert any(m.msg_type=='prompt' and m.content==TMPL_PROMPT for m in back.messages)
    test_eq(is_clean(active_items(load_rollout(sid,home))),[])
    test_eq(json.loads((_state_root()/'doced'/f'{sid}.json').read_text()),['doc','rg'])

### Command line

The CLI can build a reviewed template, import a clean completed thread, print a prepared thread id, append after compaction, or resume a new Codex thread.

In [ ]:
#| export
USAGE = """usage: codexdojo [--build <dialog.ipynb> | --capture-current [thread-id] | --sid | -r [thread-id] | <codex args...>]
Start Codex in the current project with a completed dojo round already in its session history.
  --build <dialog.ipynb>       convert a reviewed template dialog and store it
  --capture-current [thread]   curate an existing thread's clean round and store it
  --sid                        prepare and print a thread id instead of launching
  -r [thread-id]               append the round to an existing thread after compaction
  <codex args...>              anything else is passed through to `codex resume`"""

def main():
    "CLI entry point; run `codexdojo -h` for usage"
    args = sys.argv[1:]
    if args[:1] in (['-h'],['--help']) or (args[:1]==['--build'] and len(args)!=2) \
        or (args[:1]==['--capture-current'] and len(args)>2) \
        or (args[:1]==['-r'] and len(args)>2): return print(USAGE)
    if args[:1]==['--build']: return build_template(args[1])
    if args[:1]==['--capture-current']:
        capture_current(args[1] if len(args)==2 else None)
        return
    if args[:1]==['-r']: return print(asyncio.run(append_dojo(args[1] if len(args)==2 else None)))
    if args==['--sid']: return print(asyncio.run(prep_dojo()))
    os.execvp('codex',['codex','resume',asyncio.run(prep_dojo()),*args])

## Rebuilding

The packaged Codex template is a normal dialog. To refresh it, run a dojo round in a separate Codex session, convert the completed thread with `items2dlg`, curate the prompt reply with `reply2dlg`, and rebuild the reviewed notebook with `codexdojo --build`. `--capture-current` handles the simpler case where the source thread already scored clean on its first attempt.

## Cleanup

In [ ]:
shutil.rmtree(store)

## Export

In [ ]:
import nbdev; nbdev.nbdev_export()